# A6 continue-train (Colab A100) — one Drive tarball

**Upload once to Google Drive:**
`MyDrive/usu-a6/a6_continue_colab.tgz`

That archive already contains: hybrid LoRA (`hybrid_start/`), continue-train JSONL, and `gold_answers.json`.

**Then:** Runtime → A100 (80GB) → Run all.

Tuned for A100 80GB: batch 512, no grad accumulation, gradient checkpointing off (max throughput for Llama-3.2-1B LoRA).

This notebook will: unpack → install → continue-train from the hybrid adapter on the new skill data → print graded-10 answers → keyword-grade → save results back to Drive.

You only need an HF token for gated `meta-llama/Llama-3.2-1B-Instruct`.

In [ ]:
# 0) Must be Colab
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    pass
assert IN_COLAB, "Open this notebook in browser Colab (A100), not a local kernel."
print("Connected to Colab OK")

In [ ]:
# 1) GPU
import subprocess

import torch

print(subprocess.getoutput("nvidia-smi"))
assert torch.cuda.is_available(), "Pick a GPU runtime (A100)."
print("GPU=", torch.cuda.get_device_name(0))

In [ ]:
# 2) Mount Drive, unpack tarball, clone repo code, stage data
import shutil
import subprocess
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_TGZ = Path("/content/drive/MyDrive/usu-a6/a6_continue_colab.tgz")
assert DRIVE_TGZ.is_file(), (
    f"Missing {DRIVE_TGZ}. Upload a6_continue_colab.tgz to MyDrive/usu-a6/ first."
)

PACK = Path("/content/a6_continue_pack")
if PACK.exists():
    shutil.rmtree(PACK)
PACK.mkdir(parents=True)
with tarfile.open(DRIVE_TGZ, "r:gz") as tar:
    tar.extractall(PACK)

# tarball may contain a top-level folder or flat files
root = PACK
kids = [p for p in PACK.iterdir() if p.name != "." ]
if len(kids) == 1 and kids[0].is_dir():
    root = kids[0]
print("Pack root:", root)
print(sorted(p.name for p in root.iterdir())[:20])

REPO_URL = "https://github.com/IMJONEZZ/USU-LLM-Class-F26.git"
BRANCH = "talha_shahzad_6"
REPO = Path("/content/USU-LLM-Class-F26")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO)]
)

DATA = REPO / "data" / "assignment6"
QA = DATA / "qa"
QA.mkdir(parents=True, exist_ok=True)
adapter_src = root / "hybrid_start"
assert adapter_src.is_dir(), adapter_src
adapter_dst = DATA / "adapters" / "hybrid_start"
if adapter_dst.exists():
    shutil.rmtree(adapter_dst)
shutil.copytree(adapter_src, adapter_dst)
shutil.copy(root / "qa" / "chatgpt_continue_train.jsonl", QA / "chatgpt_continue_train.jsonl")
shutil.copy(root / "qa" / "chatgpt_continue_dev.jsonl", QA / "chatgpt_continue_dev.jsonl")
shutil.copy(root / "gold_answers.json", DATA / "gold_answers.json")

import os

os.chdir(REPO)
print("cwd", Path.cwd())
print("adapter files", len(list(adapter_dst.iterdir())))
!wc -l data/assignment6/qa/chatgpt_continue_*.jsonl

In [ ]:
# 3) Install deps
%pip -q install -U pip
%pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip -q install peft trl accelerate bitsandbytes datasets "transformers==4.57.6"
%pip -q install evaluate rouge-score bert-score

In [ ]:
# 4) HF login (gated Llama)
import getpass
import os

from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF token: ")
login(token=token)
os.environ["HF_TOKEN"] = token
print("HF login OK")

In [ ]:
import os
# 5) Validate JSONL + prepare-qa
import json
import shutil
import subprocess
from pathlib import Path

from src.trainer import find_graded_leakage, validate_qa_record

os.chdir("/content/USU-LLM-Class-F26")
train_path = Path("data/assignment6/qa/chatgpt_continue_train.jsonl")
dev_path = Path("data/assignment6/qa/chatgpt_continue_dev.jsonl")


def load_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(train_path)
dev_rows = load_jsonl(dev_path)
assert train_rows and dev_rows
for row in train_rows + dev_rows:
    validate_qa_record(row)
    assert row["review_status"] == "approved", row["example_id"]
leak = find_graded_leakage(train_rows + dev_rows)
assert not leak, leak[:3]
print(f"OK train={len(train_rows)} dev={len(dev_rows)}")

out_prep = Path("data/assignment6/prepared_continue")
if out_prep.exists():
    shutil.rmtree(out_prep)
subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "prepare-qa",
        "--train",
        str(train_path),
        "--dev",
        str(dev_path),
        "--output",
        str(out_prep),
    ]
)
print("prepared_continue OK")

In [ ]:
# 6) A100 80GB knobs — maximize throughput for Llama-3.2-1B LoRA
BATCH_SIZE = 512  # ~3 steps/epoch on 1430 rows; drop to 256/128 only on OOM
ACCUMULATION = 1  # real large batches beat fake accumulation on 80GB
EPOCHS = 3
FEASIBILITY_STEPS = 2  # short gate; batch already huge
print("effective_batch", BATCH_SIZE * ACCUMULATION)

In [ ]:
import os
# 7) Feasibility + continue-train from hybrid_start
import shutil
import subprocess
from pathlib import Path

os.chdir("/content/USU-LLM-Class-F26")
for d in ["data/assignment6/feasibility_continue", "data/assignment6/train_continue"]:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)

common = [
    "--prepared",
    "data/assignment6/prepared_continue",
    "--init-adapter",
    "data/assignment6/adapters/hybrid_start",
    "--batch-size",
    str(BATCH_SIZE),
    "--accumulation",
    str(ACCUMULATION),
    "--learning-rate",
    "2e-4",
    "--rank",
    "4",
    "--alpha",
    "8",
    "--targets",
    "q_proj,v_proj",
]
# Optional flag if present on this branch
help_txt = subprocess.getoutput("python -m src.trainer train -h")
if "--no-gradient-checkpointing" in help_txt or "--gradient-checkpointing" in help_txt:
    common.append("--no-gradient-checkpointing")

subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "feasibility",
        *common,
        "--output",
        "data/assignment6/feasibility_continue",
        "--steps",
        str(FEASIBILITY_STEPS),
    ]
)
subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "train",
        *common,
        "--feasibility",
        "data/assignment6/feasibility_continue",
        "--output",
        "data/assignment6/train_continue",
        "--epochs",
        str(EPOCHS),
    ]
)
assert Path("data/assignment6/train_continue/success.json").is_file()
print("TRAIN_OK")
print(subprocess.getoutput("nvidia-smi"))

In [ ]:
import os
# 8) Graded-10 answers with continued adapter
import json
from datetime import UTC, datetime
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.trainer import GRADED_QUESTIONS

os.chdir("/content/USU-LLM-Class-F26")
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER = Path("data/assignment6/train_continue/best")
OUT = Path("data/assignment6/after_continue_train.json")
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(base, ADAPTER)
model.to(DEVICE)
model.eval()


def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


examples = []
for q in GRADED_QUESTIONS:
    ans = ask(q)
    examples.append({"question": q, "answer": ans})
    print("Q:", q)
    print("A:", ans)
    print("---")

payload = {
    "model": MODEL_ID,
    "adapter": str(ADAPTER),
    "device": DEVICE,
    "max_new_tokens": 128,
    "generation": "greedy",
    "created_utc": datetime.now(UTC).isoformat(),
    "examples": examples,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
print("Wrote", OUT)

In [ ]:
import os
# 9) Keyword grade + save results pack to Drive
import json
import shutil
import tarfile
from pathlib import Path

os.chdir("/content/USU-LLM-Class-F26")
DATA = Path("data/assignment6")
gold = json.loads((DATA / "gold_answers.json").read_text())
pred = json.loads((DATA / "after_continue_train.json").read_text())


def norm(text: str) -> str:
    return " ".join(str(text).lower().split())


def keyword_pass(prediction: str, item: dict) -> bool:
    if any(norm(bad) in norm(prediction) for bad in item.get("reject_if") or []):
        return False
    keywords = item.get("keywords") or []
    hits = [kw for kw in keywords if norm(kw) in norm(prediction)]
    if item.get("match") == "all":
        return len(hits) == len(keywords)
    return bool(hits)


passed = 0
lines = ["# Continue-train graded keyword report", ""]
for item, ex in zip(gold, pred["examples"], strict=True):
    ok = keyword_pass(ex["answer"], item)
    passed += int(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"Q{item['id']}: {mark} | {ex['answer'][:100]}")
    lines.append(f"- Q{item['id']}: **{mark}** — {ex['answer']}")
lines += ["", f"**Keyword score: {passed}/{len(gold)}**", ""]
report = DATA / "after_continue_train_report.md"
report.write_text("\n".join(lines), encoding="utf-8")
print(f"Keyword {passed}/{len(gold)}")

out_dir = Path("/content/drive/MyDrive/usu-a6/results_continue")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True)
shutil.copytree(DATA / "train_continue" / "best", out_dir / "adapter_best")
shutil.copy(DATA / "after_continue_train.json", out_dir / "after_continue_train.json")
shutil.copy(report, out_dir / "after_continue_train_report.md")
shutil.copy(DATA / "train_continue" / "training.json", out_dir / "training.json")
shutil.copy(DATA / "train_continue" / "success.json", out_dir / "success.json")

results_tgz = Path("/content/drive/MyDrive/usu-a6/a6_continue_results.tgz")
with tarfile.open(results_tgz, "w:gz") as tar:
    tar.add(out_dir, arcname="results_continue")
print("Saved folder:", out_dir)
print("Saved tarball:", results_tgz)
print("DONE — download a6_continue_results.tgz when finished.")